# Vector Semantics, Cosine Similarity, PMI and PPMI

**Course:** CS F429 Natural Language Processing  
**Topic:** Vector Semantics & Embeddings  

This notebook solves Tutorial V step by step using the same formula style used in the lecture slides: term-document matrix, cosine similarity, PMI, and PPMI.

## Question

Given documents:

- Doc1: `apple apple orange mango`
- Doc2: `apple mango orange`
- Doc3: `carrot raddish carrot raddish`
- Doc4: `carrot raddish raddish`
- Doc5: `mango mango orange`

Query: `apple orange`

Tasks:

1. Construct the term-document matrix using term frequencies.
2. Compute cosine similarity between the query and each document.
3. Identify which term, `apple` or `orange`, contributes more to retrieving each document.
4. Compute PMI and PPMI.
5. Compare cosine similarity and PPMI: which captures term importance and which captures term association?

In [1]:
import math
import pandas as pd

documents = {
    "Doc1": "apple apple orange mango".split(),
    "Doc2": "apple mango orange".split(),
    "Doc3": "carrot raddish carrot raddish".split(),
    "Doc4": "carrot raddish raddish".split(),
    "Doc5": "mango mango orange".split(),
}

query_terms = "apple orange".split()
vocab = sorted(set(word for doc in documents.values() for word in doc))
print("Vocabulary:", vocab)

Vocabulary: ['apple', 'carrot', 'mango', 'orange', 'raddish']


## 1. Term-Document Matrix

Each row represents a term and each column represents a document. The cell value is the raw term frequency.

In [2]:
term_doc_matrix = pd.DataFrame({
    doc_name: {term: words.count(term) for term in vocab}
    for doc_name, words in documents.items()
}).fillna(0).astype(int)

term_doc_matrix

,Doc1,Doc2,Doc3,Doc4,Doc5
apple,2,1,0,0,0
carrot,0,0,2,1,0
mango,1,1,0,0,2
orange,1,1,0,0,1
raddish,0,0,2,2,0


## 2. Query Vector

The query is **apple orange**, so the query vector is constructed using the same vocabulary order.

In [3]:
query_vector = pd.Series({term: query_terms.count(term) for term in vocab}, name="Query")
query_vector.to_frame()

,Query
apple,1
carrot,0
mango,0
orange,1
raddish,0


## 3. Cosine Similarity

Cosine similarity is computed as:

$$
\cos(ec{q}, ec{d}) = rac{ec{q} \cdot ec{d}}{|ec{q}| |ec{d}|}
$$

The query vector is compared with each document vector.

In [4]:
def cosine_similarity(vec1, vec2):
    dot_product = sum(vec1[term] * vec2[term] for term in vocab)
    norm1 = math.sqrt(sum(vec1[term] ** 2 for term in vocab))
    norm2 = math.sqrt(sum(vec2[term] ** 2 for term in vocab))
    if norm1 == 0 or norm2 == 0:
        return 0
    return dot_product / (norm1 * norm2)

cosine_results = []
for doc_name in documents:
    doc_vector = term_doc_matrix[doc_name]
    dot = sum(query_vector[term] * doc_vector[term] for term in vocab)
    q_norm = math.sqrt(sum(query_vector[term] ** 2 for term in vocab))
    d_norm = math.sqrt(sum(doc_vector[term] ** 2 for term in vocab))
    cosine = cosine_similarity(query_vector, doc_vector)
    cosine_results.append({
        "Document": doc_name,
        "Dot product": dot,
        "|Query|": round(q_norm, 4),
        "|Document|": round(d_norm, 4),
        "Cosine Similarity": round(cosine, 4)
    })

cosine_df = pd.DataFrame(cosine_results).sort_values(by="Cosine Similarity", ascending=False)
cosine_df

,Document,Dot product,|Query|,|Document|,Cosine Similarity
0,Doc1,3,1.4142,2.4495,0.8660
1,Doc2,2,1.4142,1.7321,0.8165
4,Doc5,1,1.4142,2.2361,0.3162
2,Doc3,0,1.4142,2.8284,0.0000
3,Doc4,0,1.4142,2.2361,0.0000


### Cosine Similarity Ranking

The most relevant document for the query **apple orange** is **Doc1**, because it has the highest cosine similarity score.

In [5]:
best_doc = cosine_df.iloc[0]
print(f"Most relevant document = {best_doc['Document']} with cosine similarity {best_doc['Cosine Similarity']}")

Most relevant document = Doc1 with cosine similarity 0.866


## 4. Contribution of `apple` and `orange`

For this query, only the dimensions `apple` and `orange` contribute to the dot product.

Contribution of a term to cosine similarity is:

$$
rac{q_t 	imes d_t}{|q||d|}
$$

where $q_t$ is the query frequency of the term and $d_t$ is the document frequency of the term within that document.

In [6]:
q_norm = math.sqrt(sum(query_vector[term] ** 2 for term in vocab))
contribution_rows = []

for doc_name in documents:
    doc_vector = term_doc_matrix[doc_name]
    d_norm = math.sqrt(sum(doc_vector[term] ** 2 for term in vocab))
    for term in ["apple", "orange"]:
        raw_contribution = query_vector[term] * doc_vector[term]
        cosine_contribution = raw_contribution / (q_norm * d_norm) if d_norm != 0 else 0
        contribution_rows.append({
            "Document": doc_name,
            "Term": term,
            "Term count in document": int(doc_vector[term]),
            "Raw dot contribution": raw_contribution,
            "Cosine contribution": round(cosine_contribution, 4)
        })

contribution_df = pd.DataFrame(contribution_rows)
contribution_df

,Document,Term,Term count in document,Raw dot contribution,Cosine contribution
0,Doc1,apple,2,2,0.5774
1,Doc1,orange,1,1,0.2887
2,Doc2,apple,1,1,0.4082
3,Doc2,orange,1,1,0.4082
4,Doc3,apple,0,0,0.0000
5,Doc3,orange,0,0,0.0000
6,Doc4,apple,0,0,0.0000
7,Doc4,orange,0,0,0.0000
8,Doc5,apple,0,0,0.0000
9,Doc5,orange,1,1,0.3162


In [7]:
for doc_name in documents:
    sub = contribution_df[contribution_df["Document"] == doc_name]
    apple_score = float(sub[sub["Term"] == "apple"]["Cosine contribution"].iloc[0])
    orange_score = float(sub[sub["Term"] == "orange"]["Cosine contribution"].iloc[0])
    if apple_score > orange_score:
        print(f"{doc_name}: apple contributes more than orange.")
    elif orange_score > apple_score:
        print(f"{doc_name}: orange contributes more than apple.")
    elif apple_score == orange_score and apple_score != 0:
        print(f"{doc_name}: apple and orange contribute equally.")
    else:
        print(f"{doc_name}: neither apple nor orange contributes; cosine similarity is 0.")

Doc1: apple contributes more than orange.
Doc2: apple and orange contribute equally.
Doc3: neither apple nor orange contributes; cosine similarity is 0.
Doc4: neither apple nor orange contributes; cosine similarity is 0.
Doc5: orange contributes more than apple.


## 5. PMI and PPMI

The lecture formula for PMI is:

$$
PMI(w,c) = \log_2 rac{P(w,c)}{P(w)P(c)}
$$

For this tutorial, we treat each document as a context. Therefore:

$$
P(w,c) = rac{f_{w,c}}{N}
$$

$$
P(w) = rac{\sum_c f_{w,c}}{N}
$$

$$
P(c) = rac{\sum_w f_{w,c}}{N}
$$

where $N$ is the total number of tokens in the corpus.

PPMI replaces negative PMI values with 0:

$$
PPMI(w,c) = \max(PMI(w,c), 0)
$$

In [8]:
F = term_doc_matrix.copy()
N = int(F.values.sum())
row_totals = F.sum(axis=1)
col_totals = F.sum(axis=0)

print("Total token count N =", N)
print("\nRow totals P(w) numerator:")
print(row_totals)
print("\nColumn totals P(c) numerator:")
print(col_totals)

Total token count N = 17

Row totals P(w) numerator:
apple      3
carrot     3
mango      4
orange     3
raddish    4
dtype: int64

Column totals P(c) numerator:
Doc1    4
Doc2    3
Doc3    4
Doc4    3
Doc5    3
dtype: int64


In [9]:
pmi_matrix = pd.DataFrame(index=vocab, columns=documents.keys(), dtype=float)
ppmi_matrix = pd.DataFrame(index=vocab, columns=documents.keys(), dtype=float)

for term in vocab:
    for doc_name in documents:
        f_wc = F.loc[term, doc_name]
        if f_wc == 0:
            pmi_matrix.loc[term, doc_name] = float("-inf")
            ppmi_matrix.loc[term, doc_name] = 0.0
        else:
            p_wc = f_wc / N
            p_w = row_totals[term] / N
            p_c = col_totals[doc_name] / N
            pmi_value = math.log2(p_wc / (p_w * p_c))
            pmi_matrix.loc[term, doc_name] = pmi_value
            ppmi_matrix.loc[term, doc_name] = max(pmi_value, 0)

pmi_matrix.replace(float("-inf"), "-inf").round(4)

,Doc1,Doc2,Doc3,Doc4,Doc5
apple,1.5025,0.917538,-inf,-inf,-inf
carrot,-inf,-inf,1.5025,0.917538,-inf
mango,0.087463,0.5025,-inf,-inf,1.5025
orange,0.5025,0.917538,-inf,-inf,0.917538
raddish,-inf,-inf,1.087463,1.5025,-inf


### PPMI Matrix

Negative PMI values are replaced by 0.

In [10]:
ppmi_matrix.round(4)

,Doc1,Doc2,Doc3,Doc4,Doc5
apple,1.5025,0.9175,0.0000,0.0000,0.0000
carrot,0.0000,0.0000,1.5025,0.9175,0.0000
mango,0.0875,0.5025,0.0000,0.0000,1.5025
orange,0.5025,0.9175,0.0000,0.0000,0.9175
raddish,0.0000,0.0000,1.0875,1.5025,0.0000


## 6. PPMI-Based Query Relevance

To compare with cosine similarity, we can sum the PPMI scores of the query terms `apple` and `orange` for each document.

In [11]:
ppmi_query_score = ppmi_matrix.loc[["apple", "orange"]].sum(axis=0).sort_values(ascending=False)
ppmi_query_score.to_frame(name="PPMI query score")

,PPMI query score
Doc1,2.005001
Doc2,1.835076
Doc5,0.917538
Doc3,0.000000
Doc4,0.000000


## 7. Final Comparison

### Cosine Similarity

Cosine similarity compares the query vector with each document vector. It captures **document relevance to the query** based on vector direction. It is useful for ranking documents because it considers the combined presence of the query terms and normalizes for document length.

For the query **apple orange**, cosine similarity retrieves the documents containing both query terms most strongly.

### PPMI

PPMI captures **term-context association**. It tells us whether a word occurs in a document more strongly than expected by chance. A high PPMI value means that the term is highly associated with that document context.

### Which is better?

- For **retrieving the most relevant document for a query**, cosine similarity is more direct.
- For **understanding term association with a document/context**, PPMI is better.
- Cosine similarity captures overall query-document similarity.
- PPMI captures how informative or strongly associated a term is with a context.

### Final Answer

The most relevant document using cosine similarity is **Doc1**, followed by **Doc2**. Doc5 is relevant only through `orange`, while Doc3 and Doc4 are not relevant to the query because they contain neither `apple` nor `orange`.